# 02/ Model Training - Ridge Regression

This notebook trains a Ridge Regression model on the CMAPSS dataset to predict the Remaining Useful Life (RUL) of aircraft engines. It includes hyperparameter tuning via cross-validated grid search, performance evaluation, and feature coefficient analysis.

### 1 - Setup

Import system utilities, custom data loading and evaluation functions, and all required libraries for modelling, preprocessing, cross-validation, and visualisation.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
    
from src.data_processing import load_train_data
from src.data_processing import load_test_data
from src.model_evaluation import evaluate
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### 2 - Data Preparation

Load train and test datasets, then compute the variance of each sensor column. Features with zero variance (constant across all cycles) are discarded as non-informative. The remaining high-variance features are used to build the final feature matrices X_train and X_test.

In [ ]:
df_train = load_train_data("../CMAPSSData/train_FD001.txt")

df_test = load_test_data("../CMAPSSData/test_FD001.txt", "../CMAPSSData/RUL_FD001.txt")

variance_table = df_train[df_train.columns[5:26]]
variance_table = variance_table.var()
variance_table = variance_table.to_frame(name='Variance')
variance_table = variance_table.round(5)
high_variance_features = variance_table[variance_table['Variance'] != 0].index.tolist()

X_train = df_train[high_variance_features]
y_train = df_train['RUL']

X_test = df_test[high_variance_features]
y_test = df_test['RUL']

### 3 - Ridge

Build a pipeline combining standard scaling and Ridge Regression. Use Group K-Fold cross-validation (grouped by engine unit) to avoid data leakage across engines. Run a grid search over a range of alpha values to find the regularisation strength minimising the cross-validated RMSE. Plot the RMSE curve against alpha and mark the optimal value.
Then evaluate the best model on two test sets: all test rows, and only the last cycle per engine (the official NASA CMAPSS evaluation protocol). Finally, inspect the model coefficients ranked by absolute magnitude to identify the most influential features.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=42))
])

groups = df_train['unit']
gkf = GroupKFold(n_splits=5)

# param_grid={'ridge__alpha': [i for i in range(0, 10000, 100)]}
# param_grid={'ridge__alpha': [i for i in range(400, 600, 10)]}
# param_grid={'ridge__alpha': [i for i in range(530, 550)]}
param_grid = {'ridge__alpha': [i for i in range(345, 745, 20)]}

ridge_cv = GridSearchCV(
    pipeline,
    param_grid=param_grid,  
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1   
)
ridge_cv.fit(X_train, y_train, groups=groups)

alphas_testes = np.array(ridge_cv.cv_results_['param_ridge__alpha'], dtype=float)
cv_rmse = -ridge_cv.cv_results_['mean_test_score'] 
best_alpha = ridge_cv.best_params_['ridge__alpha']
best_score = -ridge_cv.best_score_

plt.figure(figsize=(10, 5))
plt.plot(alphas_testes, cv_rmse, marker='o', linestyle='-', color='darkblue', markersize=4, label='CV RMSE')
plt.axvline(x=best_alpha, color='crimson', linestyle='--', linewidth=1.5, 
            label=f'Best Alpha : {best_alpha:.2f} (RMSE: {best_score:.4f})')
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('RMSE')
plt.title('Alpha Optimisation for Ridge Regression')
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()

In [ ]:
# Predictions for all test rows
y_pred_all = ridge_cv.best_estimator_.predict(X_test)
# RMSE and official SCORE metric for all test rows
evaluate(y_test, y_pred_all, label="All test rows")

# Last cycle per engine 
last_idx = df_test.groupby("unit")["cycle"].idxmax()
# Test data for the last cycle of each engine
X_test_last = df_test.loc[last_idx, high_variance_features]
# True RUL values for the last cycle of each engine
y_test_last = df_test.loc[last_idx, "RUL"]

# Predictions for the last cycle of each engine
y_pred_last = ridge_cv.best_estimator_.predict(X_test_last)
# RMSE and official SCORE metric for the last cycle of each engine
evaluate(y_test_last, y_pred_last, label="Last cycle per engine (official)")

In [ ]:
feature_names = high_variance_features
coefs = ridge_cv.best_estimator_.named_steps['ridge'].coef_

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
print(coef_df)